<center>    
<span style="font-family: Arial; font-weight:bold;font-size:2.75em;color:#9F1F1B;">SIG731 Week 4 Master class - Part 2 :

#### Accessing SQL using Pandas
#### (Mapping Pandas functions against SQL Operations)

In [28]:
import pandas as pd

import numpy as np

### Import the tips dataset

In [29]:
url = ("https://raw.githubusercontent.com/pandas-dev/pandas/main/pandas/tests/io/data/csv/tips.csv" )

tips = pd.read_csv(url)

tips

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


### Estabish a new SQL connection, with a new SQLite database

In [31]:
import tempfile, os.path
dbfile = os.path.join(tempfile.mkdtemp(), "tips.db")
print(dbfile)

/tmp/tmpzowvfw0b/tips.db


In [32]:
import sqlite3
conn = sqlite3.connect(dbfile)

### Export data from the dataframe to database

In [33]:
tips.to_sql("tips", conn, index=False)

244

### Accessing the database from Pandas

#### 1. Retrieve the top 3 rows from the table

In [34]:
# run the sql query

res1a = pd.read_sql_query("""
    SELECT * FROM tips LIMIT 3
""", conn)

res1a

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3


In [38]:
res1a.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  3 non-null      float64
 1   tip         3 non-null      float64
 2   sex         3 non-null      object 
 3   smoker      3 non-null      object 
 4   day         3 non-null      object 
 5   time        3 non-null      object 
 6   size        3 non-null      int64  
dtypes: float64(2), int64(1), object(4)
memory usage: 300.0+ bytes


In [35]:
# equivalent pandas query
res1b = tips.head(3)
res1b

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3


In [36]:
# verify that the results are equal:
pd.testing.assert_frame_equal(res1a, res1b)  # no error == OK

#### 2. From Tags, select two columns day and tip and rows for which day is equal to one of the three choices provided. [ Sat, Sun, Fri]

In [9]:
res2a = pd.read_sql_query("""
    SELECT day, tip
    FROM tips
    WHERE day IN ('Sat', 'Sun', 'Fri')
""", conn)

res2a

,day,tip
0,Sun,1.01
1,Sun,1.66
2,Sun,3.50
3,Sun,3.31
4,Sun,3.61
...,...,...
177,Sat,4.67
178,Sat,5.92
179,Sat,2.00
180,Sat,2.00


In [40]:
res2b = (
    tips.
    loc[
        tips.day.isin(["Sat", "Sun", "Fri"]),
        ["day", "tip"]
    ]
)

res2b

,day,tip
0,Sun,1.01
1,Sun,1.66
2,Sun,3.50
3,Sun,3.31
4,Sun,3.61
...,...,...
238,Sat,4.67
239,Sat,5.92
240,Sat,2.00
241,Sat,2.00


In [10]:
# equivalent pandas query
res2b = (
    tips.
    loc[
        tips.day.isin(["Sat", "Sun", "Fri"]),
        ["day", "tip"]
    ].
    reset_index(drop=True)
)

In [11]:
# verify that the results are equal:
pd.testing.assert_frame_equal(res2a, res2b)  # no error == OK

#### Select a set of columns from tips whose rows fulfil a given set of conditions. time = 'Lunch' AND tip >= 2.5

In [12]:
tips.describe(include = 'all')

,total_bill,tip,sex,smoker,day,time,size
count,244.000000,244.000000,244,244,244,244,244.000000
unique,NaN,NaN,2,2,4,2,NaN
top,NaN,NaN,Male,No,Sat,Dinner,NaN
freq,NaN,NaN,157,151,87,176,NaN
mean,19.785943,2.998279,NaN,NaN,NaN,NaN,2.569672
std,8.902412,1.383638,NaN,NaN,NaN,NaN,0.951100
min,3.070000,1.000000,NaN,NaN,NaN,NaN,1.000000
25%,13.347500,2.000000,NaN,NaN,NaN,NaN,2.000000
50%,17.795000,2.900000,NaN,NaN,NaN,NaN,2.000000
75%,24.127500,3.562500,NaN,NaN,NaN,NaN,3.000000


In [13]:
tips.dtypes

,0
total_bill,float64
tip,float64
sex,object
smoker,object
day,object
time,object
size,int64


In [14]:
res3a = pd.read_sql_query("""
    SELECT total_bill, tip, day, size
    FROM tips
    WHERE time = 'Lunch' AND
        tip >= 2.5

""", conn)
res3a

,total_bill,tip,day,size
0,27.20,4.00,Thur,4
1,22.76,3.00,Thur,2
2,17.29,2.71,Thur,2
3,19.44,3.00,Thur,2
4,16.66,3.40,Thur,2
5,32.68,5.00,Thur,2
6,34.83,5.17,Thur,4
7,18.28,4.00,Thur,2
8,24.71,5.85,Thur,2
9,21.16,3.00,Thur,2


In [41]:
res3b = (
    tips.
    loc[
        (tips.time == 'Lunch') & (tips.tip >= 2.5)
        & (tips.size >= 4),
        ['total_bill', 'tip', 'day', 'size']
    ].
    reset_index(drop=True)
)

res3b

,total_bill,tip,day,size
0,27.20,4.00,Thur,4
1,22.76,3.00,Thur,2
2,17.29,2.71,Thur,2
3,19.44,3.00,Thur,2
4,16.66,3.40,Thur,2
5,32.68,5.00,Thur,2
6,34.83,5.17,Thur,4
7,18.28,4.00,Thur,2
8,24.71,5.85,Thur,2
9,21.16,3.00,Thur,2


In [16]:
pd.testing.assert_frame_equal(res3a, res3b)  # no error == OK

#### For each male smoker with a total bill exceeding 30; extract the tip, store it in a new column named m_smoker_tip. Then, select only the unique pairs (bill, m_smoker_tip).

In [17]:
res4a = pd.read_sql_query("""
    SELECT DISTINCT total_bill, tip as m_smoker_tip
    FROM tips
    WHERE sex = 'Male'
    and total_bill > 30
    and smoker = 'Yes'
""", conn)
res4a

,total_bill,m_smoker_tip
0,38.01,3.00
1,32.68,5.00
2,40.17,4.73
3,50.81,10.00
4,31.85,3.18
5,32.90,3.11
6,34.63,3.55
7,34.65,3.68
8,45.35,3.50
9,40.55,3.00


In [42]:
tips[(tips.sex == 'Male') & (tips.smoker == 'Yes')& (tips.total_bill > 30)][['total_bill', 'tip']].drop_duplicates().reset_index(drop=True).rename_columns

,total_bill,tip
0,38.01,3.00
1,32.68,5.00
2,40.17,4.73
3,50.81,10.00
4,31.85,3.18
5,32.90,3.11
6,34.63,3.55
7,34.65,3.68
8,45.35,3.50
9,40.55,3.00


In [18]:
res4b = (
    tips.
    loc[
        (tips.sex == 'Male') & (tips.smoker == 'Yes')
        & (tips.total_bill > 30),
        ['total_bill', 'tip']].
    drop_duplicates().
    reset_index(drop=True)
)

res4b.rename(columns = {'tip' : 'm_smoker_tip'}, inplace = True) #rename column

pd.testing.assert_frame_equal(res4a, res4b)  # no error == OK

#### Count how many male smokers had a total bill exceeding 30 Dollars everyday. Also, for each day, compute the minimal, average, and maximal tip received. Return only the top 3 days  (with respect to the average tips received).

In [19]:
res5a = pd.read_sql_query("""
    SELECT DISTINCT day, count(*) "count", min(tip) min_tip, avg(tip) avg_tip, max(tip) max_tip
    FROM tips
    WHERE sex = 'Male'
    and total_bill > 30
    and smoker = 'Yes'
    group by day
    order by avg_tip desc
    limit 3
""", conn)
res5a

,day,count,min_tip,avg_tip,max_tip
0,Thur,1,5.00,5.000,5.00
1,Fri,1,4.73,4.730,4.73
2,Sat,5,1.17,3.834,10.00


In [20]:
res5b = (
    tips.
    loc[(tips.sex == 'Male') & (tips.smoker == 'Yes')
        & (tips.total_bill > 30), ['day', "total_bill", "tip"]].
    groupby("day")["tip"].
    aggregate(["count", "min", "mean", "max"]).
    sort_values("mean", ascending=False).
    head(3).
    reset_index()
)

res5b.columns = ["day", 'count', "min_tip", "avg_tip", "max_tip"]

In [21]:
pd.testing.assert_frame_equal(res5a, res5b)  # no error == OK

## Close the connection to database

In [22]:
conn.close()